In [ ]:
import pandas as pd

df_data = pd.read_csv('pilot.csv')
df_data

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# Set your directory path
image_dir = 'data/pilot'

# Collect all image paths matching pattern
image_data = []

for filename in os.listdir(image_dir):
    if filename.startswith("left_") and filename.endswith(".png"):
        # Extract timestamp from filename
        timestamp = filename[5:-4]  # Removes 'left_' and '.png'
        
        # Read the image
        img_path = os.path.join(image_dir, filename)
        image_np = cv2.imread(img_path)  # Read as BGR numpy array

        # Append to list
        image_data.append((timestamp, image_np))

# Create the DataFrame
df_img = pd.DataFrame(image_data, columns=['timestamp', 'image'])

# Optional: sort by timestamp
df_img = df_img.sort_values(by='timestamp').reset_index(drop=True)


df_img

In [ ]:
# Ensure timestamps are float and round to 1 decimal
df_img['timestamp'] = df_img['timestamp'].astype(float).round(1)
df_data['timestamp'] = df_data['timestamp'].astype(float).round(1)

# Merge: many df_data rows can match one image
df_merged = pd.merge(df_data, df_img, on='timestamp', how='left')


# Drop rows with null (NaN) in the 'image' column
df_merged = df_merged[df_merged['image'].notna()]


df_merged

In [ ]:

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from scipy.optimize import least_squares

# ─── 0) Load & clean your DataFrame ──────────────────────────────
df = df_merged.copy().dropna(subset=[
    'x','y','z','roll','pitch','yaw','fx','fy','fz'
])
n = len(df)
poses = df[['x','y','z','roll','pitch','yaw']].to_numpy()

# turn every signal into a NumPy array
y     = df['y'].to_numpy()
fx    = df['fx'].to_numpy()
fy    = df['fy'].to_numpy()
fz    = df['fz'].to_numpy()
roll  = df['roll'].to_numpy()
pitch = df['pitch'].to_numpy()
yaw   = df['yaw'].to_numpy()

# ─── 1) Compute Flow (F), Optimality (O) & Rewards ──────────────
ymin, ymax = y.min(), y.max()
F = np.full(n, 0.5) if np.isclose(ymin, ymax) else np.interp(y, (ymin, ymax), (0, 1))

signals = [y, fx, fy, fz, roll, pitch, yaw]
jerks = np.vstack([np.abs(sig[1:] - sig[:-1]) for sig in signals])
cj = jerks.mean(axis=0)
if np.isclose(cj.min(), cj.max()):
    norm = np.zeros_like(cj)
else:
    norm = (cj - cj.min()) / (cj.max() - cj.min())
O = np.concatenate(([1.0], 1 - norm))             # prepend to length n

rewards     = np.round(200 * F + O - 100, 2)      # centered around 0
colors_rgba = np.hstack([np.stack([1-F, F, O],1), np.ones((n,1))])

# ─── 1.1) Save dataset + rewards to a DataFrame ───────────────
df_results = df.copy()
df_results['Flow']       = F
df_results['Optimality'] = O
df_results['Reward']     = rewards
# Now df_results holds your original data plus Flow, Optimality, Reward
# ─── 1.2) Export to CSV ────────────────────────────────────────
df_results.to_csv('df_results.csv', index=False)


In [ ]:
%matplotlib inline

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from scipy.optimize import least_squares
# ─── 2) DH, FK, IK & precompute skeleton positions ──────────────
PSM_DH = [
    [-math.pi/2, 0.0, 0.0, 0.0, 'R'],
    [ math.pi/2, 0.0, 0.0, 0.0, 'R'],
    [      0.0,  0.0, 0.0, 0.0, 'P'],
    [-math.pi/2, 0.0, 0.0, 0.0, 'R'],
    [ math.pi/2, 0.0, 0.0, 0.0, 'R'],
    [-math.pi/2, 0.0, 0.0, 0.0, 'R'],
    [      0.0,  0.0, 0.0, 0.0, 'R'],
]

def dh(a, alpha, d, theta):
    return np.array([
        [ math.cos(theta), -math.sin(theta)*math.cos(alpha),
          math.sin(theta)*math.sin(alpha), a*math.cos(theta)],
        [ math.sin(theta),  math.cos(theta)*math.cos(alpha),
         -math.cos(theta)*math.sin(alpha), a*math.sin(theta)],
        [           0.0,               math.sin(alpha),
                    math.cos(alpha),               d],
        [           0.0,                        0.0,
                           0.0,            1.0]
    ])

def fk(q):
    T = np.eye(4)
    for i, (alpha, a, d0, th0, jt) in enumerate(PSM_DH):
        if jt=='R':
            theta, d = th0+q[i], d0
        elif jt=='P':
            theta, d = th0, d0+q[i]
        else:
            theta, d = th0, d0
        T = T @ dh(a, alpha, d, theta)
    R = T[:3,:3]
    x, y_, z = T[0,3], T[1,3], T[2,3]
    roll_  = math.atan2(R[2,1], R[2,2])
    pitch_ = math.atan2(-R[2,0], math.hypot(R[0,0],R[1,0]))
    yaw_   = math.atan2(R[1,0], R[0,0])
    return np.array([x, y_, z, roll_, pitch_, yaw_])

def ik_err(q, tgt):
    return fk(q) - tgt

# Solve IK
qs = np.zeros((n,7))
q0 = np.zeros(7)
for i, tgt in enumerate(poses):
    sol = least_squares(ik_err, q0, args=(tgt,), xtol=1e-6, ftol=1e-6)
    qs[i], q0 = sol.x, sol.x

def joint_positions(q):
    T = np.eye(4)
    pts = [(0.0,0.0,0.0)]
    for i, (alpha, a, d0, th0, jt) in enumerate(PSM_DH):
        if jt=='R':
            theta, d = th0+q[i], d0
        elif jt=='P':
            theta, d = th0, d0+q[i]
        else:
            theta, d = th0, d0
        T = T @ dh(a, alpha, d, theta)
        pts.append((T[0,3], T[1,3], T[2,3]))
    return np.array(pts)

jtraj = [joint_positions(q) for q in qs]

# ─── 3) Inline plotting with coloured trace & live reward ────────
xv, yv, zv = poses[:,0], poses[:,1], poses[:,2]

plt.ion()
fig = plt.figure(figsize=(7,6))
ax  = fig.add_subplot(111, projection='3d')
ax.set_xlim(xv.min(), xv.max())
ax.set_ylim(yv.min(), yv.max())
ax.set_zlim(zv.min(), zv.max())

# one scatter for all past points
scatter_all = ax.scatter([], [], [], s=20, depthshade=False)

# current dot, skeleton and reward text
dot_scatter = ax.scatter([], [], [], s=80, depthshade=False)
robot_line  , = ax.plot([], [], [], c='r', marker='o')
reward_txt  = ax.text(xv[0], yv[0], zv[0],
                      f"Reward: {rewards[0]:.2f}",
                      color='black')

for i in range(n):
    # update all past points with their colors
    scatter_all._offsets3d = (xv[:i+1], yv[:i+1], zv[:i+1])
    scatter_all.set_facecolor(colors_rgba[:i+1])
    scatter_all.set_edgecolor(colors_rgba[:i+1])

    # update moving dot
    dot_scatter._offsets3d   = ([xv[i]], [yv[i]], [zv[i]])
    dot_scatter._facecolor3d = [colors_rgba[i]]
    dot_scatter._edgecolor3d = [colors_rgba[i]]

    # update skeleton
    pts = jtraj[i]
    robot_line.set_data(pts[:,0], pts[:,1])
    robot_line.set_3d_properties(pts[:,2])

    # update reward annotation
    reward_txt.set_position((xv[i], yv[i]))
    reward_txt.set_3d_properties(zv[i], zdir='z')
    reward_txt.set_text(f"Reward: {rewards[i]:.2f}")

    clear_output(wait=True)
    display(fig)
    plt.pause(0.0001)

plt.ioff()


In [ ]:
df_results

In [ ]:
# %%
# Cell 1 – Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import gymnasium as gym
from gymnasium import spaces
from torch.distributions import Normal
from tqdm import trange
import matplotlib.pyplot as plt


In [ ]:
# %%
# Cell 2 – Dataset: numeric + image, reuse "same" images until they change
class ConvImageDiffusionDataset(Dataset):
    def __init__(self, df,
                 state_features, action_features,
                 image_col='image',
                 img_diff_threshold=1e-2,
                 state_channels=2, state_dim_per_channel=8,
                 img_size=(64,64)):
        self.state_features  = state_features
        self.action_features = action_features

        # normalize numeric states & actions
        S = df[state_features].values.astype(np.float32)
        A = df[action_features].values.astype(np.float32)
        self.s_mean, self.s_std = S.mean(0), S.std(0) + 1e-6
        self.a_mean, self.a_std = A.mean(0), A.std(0) + 1e-6
        S = (S - self.s_mean) / self.s_std
        A = (A - self.a_mean) / self.a_std
        self.states  = torch.FloatTensor(S)
        self.actions = torch.FloatTensor(A)

        # process images, only keep a new one when it changes
        raw = df[image_col].tolist()
        kept, prev = [], None
        for im in raw:
            arr = np.array(im, dtype=np.float32)/255.0
            if prev is None or np.mean(np.abs(arr - prev)) > img_diff_threshold:
                prev = arr
            kept.append(prev)
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(img_size),
        ])
        self.images = [self.transform(im) for im in kept]

        # reshape numeric into [C, L] for 1D‐CNN
        N, F = self.states.shape
        needed = state_channels * state_dim_per_channel
        if F < needed:
            pad = torch.zeros(N, needed - F)
            self.states = torch.cat([self.states, pad], dim=1)
        self.states = self.states.view(N, state_channels, -1)
        if self.states.size(2) != state_dim_per_channel:
            self.states = F.interpolate(
                self.states, size=state_dim_per_channel,
                mode='linear', align_corners=False
            )

    def __len__(self):
        return len(self.actions)

    def __getitem__(self, idx):
        return {
            'numeric': self.states[idx],  # [2×L]
            'image':   self.images[idx],  # [3×H×W]
            'action':  self.actions[idx], # [9]
        }

    def get_state_image(self, idx):
        return (
            self.states[idx].unsqueeze(0),
            self.images[idx].unsqueeze(0)
        )


In [ ]:
# %%
# Cell 3 – Environment: step through DataFrame, return gym rewards
class ConvDiffusionEnv(gym.Env):
    def __init__(self, df, state_features, action_features):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.state_features  = state_features
        self.action_features = action_features
        self.max_steps = len(df) - 1
        self.current_step = 0

        lows  = df[state_features].min().values.astype(np.float32)
        highs = df[state_features].max().values.astype(np.float32)
        self.observation_space = spaces.Box(lows, highs, dtype=np.float32)

        lows  = df[action_features].min().values.astype(np.float32)
        highs = df[action_features].max().values.astype(np.float32)
        self.action_space = spaces.Box(lows, highs, dtype=np.float32)

    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)
        self.current_step = 0
        state = self.df[self.state_features].iloc[0].values.astype(np.float32)
        return state, {}

    def step(self, action):
        action = np.clip(action,
                         self.action_space.low,
                         self.action_space.high)
        reward = float(self.df['Reward'].iloc[self.current_step])
        self.current_step += 1
        done = self.current_step >= self.max_steps
        idx = min(self.current_step, self.max_steps)
        obs = self.df[self.state_features].iloc[idx].values.astype(np.float32)
        return obs, reward, done, False, {}

In [ ]:
# %%
# Cell 4 – Policy: two‐branch CNN ⇒ Gaussian + value ⇒ PPO
class RIDLPolicy(nn.Module):
    def __init__(self,
                 num_channels, num_length,
                 img_channels=3, img_size=(64,64),
                 action_dim=9, hidden_dim=256):
        super().__init__()
        # numeric 1D branch
        self.num_branch = nn.Sequential(
            nn.Conv1d(num_channels, 32, 3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1),           nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, 3, padding=1),           nn.ReLU(),
        )
        flat_num = 64 * (num_length // 2)

        # image 2D branch
        self.img_branch = nn.Sequential(
            nn.Conv2d(img_channels, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),           nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1),           nn.ReLU(),
        )
        h, w = img_size
        flat_img = 32 * (h//4) * (w//4)

        # shared FC
        self.fc_shared = nn.Sequential(
            nn.Linear(flat_num + flat_img, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.mean_head  = nn.Linear(hidden_dim // 2, action_dim)
        self.value_head = nn.Linear(hidden_dim // 2, 1)
        self.log_std    = nn.Parameter(torch.zeros(action_dim))

    def forward(self, numeric, image):
        bn = self.num_branch(numeric).view(numeric.size(0), -1)
        bi = self.img_branch(image).view(image.size(0), -1)
        x  = self.fc_shared(torch.cat([bn, bi], dim=1))
        μ     = self.mean_head(x)
        σ     = torch.exp(self.log_std).unsqueeze(0).expand_as(μ)
        value = self.value_head(x).squeeze(-1)
        return μ, σ, value


In [ ]:
# %%
# Cell 5 – Rollout Buffer + GAE for PPO
class RolloutBuffer:
    def __init__(self, capacity, device):
        self.device = device
        self.capacity = capacity
        self.clear()

    def add(self, num, img, act, logp, val, rew, done):
        self.numerics.append(num)
        self.images.append(img)
        self.actions.append(act)
        self.logps.append(logp)
        self.values.append(val)
        self.rewards.append(rew)
        self.dones.append(done)

    def compute_returns_and_advantages(self, last_value, gamma, lam):
        vals = self.values + [last_value]
        advs, gae = [], 0
        for i in reversed(range(len(self.rewards))):
            mask = 1.0 - float(self.dones[i])
            delta = self.rewards[i] + gamma * vals[i+1] * mask - vals[i]
            gae = delta + gamma * lam * mask * gae
            advs.insert(0, gae)
        self.advantages = torch.tensor(advs, device=self.device, dtype=torch.float32)
        self.returns    = self.advantages + torch.tensor(self.values, device=self.device)
        self.numerics = torch.cat(self.numerics, dim=0)
        self.images   = torch.cat(self.images,   dim=0)
        self.actions  = torch.cat(self.actions,  dim=0)
        self.logps    = torch.cat(self.logps,    dim=0)
        self.values   = torch.tensor(self.values, device=self.device)
        self.dones    = torch.tensor(self.dones,  device=self.device, dtype=torch.float32)

    def clear(self):
        self.numerics = []
        self.images   = []
        self.actions  = []
        self.logps    = []
        self.values   = []
        self.rewards  = []
        self.dones    = []


In [ ]:
# %%
# Cell 6 – Imitation Pre-Training with loss tracking
state_features  = ['Flow','Optimality']
action_features = ['x','y','z','roll','pitch','yaw','fx','fy','fz']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# build dataset & loader
dataset = ConvImageDiffusionDataset(
    df_results, state_features, action_features,
    image_col='image', img_diff_threshold=0.01,
    state_channels=2, state_dim_per_channel=8,
    img_size=(64,64)
)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# init policy & optimizer
policy = RIDLPolicy(
    num_channels=2, num_length=8,
    img_channels=3, img_size=(64,64),
    action_dim=len(action_features),
    hidden_dim=256
).to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)

# track imitation losses
imitation_loss_history = []
for ep in range(1, 21):
    total_loss = 0
    for batch in loader:
        num = batch['numeric'].to(device)
        img = batch['image'].to(device)
        act = batch['action'].to(device)
        μ, σ, _ = policy(num, img)
        loss = F.mse_loss(μ, act)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    imitation_loss_history.append(avg_loss)
    print(f"[Imit {ep:2d}/20] MSE={avg_loss:.6f}")

In [ ]:
# %%
# Cell 7 – PPO Fine-Tuning with metric tracking
env = ConvDiffusionEnv(df, state_features, action_features)
buffer = RolloutBuffer(capacity=2000, device=device)

clip_eps, gamma, lam = 0.2, 0.99, 0.95
optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)

# track PPO metrics
ppo_reward_history = []
ppo_policy_loss_history = []
ppo_value_loss_history = []
ppo_entropy_history = []

for ppo_iter in range(1, 51):
    state, _ = env.reset()
    ep_reward = 0
    buffer.clear()

    # collect rollout
    for _ in range(buffer.capacity):
        idx = env.current_step
        num_t, img_t = dataset.get_state_image(idx)
        num_t, img_t = num_t.to(device), img_t.to(device)
        with torch.no_grad():
            μ, σ, val = policy(num_t, img_t)
            dist = Normal(μ, σ)
            act = dist.sample()
            logp = dist.log_prob(act).sum(dim=-1)
        obs, rew, done, _, _ = env.step(act.cpu().numpy().flatten())
        buffer.add(num_t, img_t, act, logp, val.item(), rew, done)
        ep_reward += rew
        if done:
            env.reset()

    # bootstrap final value
    with torch.no_grad():
        _, _, last_val = policy(*[t.to(device) for t in dataset.get_state_image(env.current_step)])
    buffer.compute_returns_and_advantages(last_val.item(), gamma, lam)

    # PPO update and metric accumulation
    p_loss_sum = v_loss_sum = ent_loss_sum = 0.0
    update_count = 0
    for _ in range(10):
        idxs = torch.randperm(buffer.capacity, device=device)
        for start in range(0, buffer.capacity, 64):
            mb = idxs[start:start+64]
            num_mb   = buffer.numerics[ mb ]
            img_mb   = buffer.images[   mb ]
            act_mb   = buffer.actions[  mb ]
            old_lp   = buffer.logps[    mb ]
            returns  = buffer.returns[  mb ]
            advs     = buffer.advantages[mb]

            μ, σ, val = policy(num_mb, img_mb)
            dist = Normal(μ, σ)
            new_lp = dist.log_prob(act_mb).sum(dim=-1)
            ratio  = torch.exp(new_lp - old_lp)

            s1 = ratio * advs
            s2 = torch.clamp(ratio,1-clip_eps,1+clip_eps)*advs
            p_loss  = -torch.min(s1, s2).mean()
            v_loss  = F.mse_loss(val, returns)
            e_loss  = -dist.entropy().sum(dim=-1).mean()
            loss    = p_loss + 0.5*v_loss + 0.01*e_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            p_loss_sum    += p_loss.item()
            v_loss_sum    += v_loss.item()
            ent_loss_sum  += e_loss.item()
            update_count  += 1

    # record metrics
    avg_ep_reward = ep_reward / buffer.capacity
    ppo_reward_history.append(avg_ep_reward)
    ppo_policy_loss_history.append(p_loss_sum/update_count)
    ppo_value_loss_history.append(v_loss_sum/update_count)
    ppo_entropy_history.append(ent_loss_sum/update_count)
    print(f"[PPO {ppo_iter:02d}/50] AvgEpReward={avg_ep_reward:.3f}")

In [ ]:

# %%
# Cell 8 – Save Final Policy
torch.save(policy.state_dict(), "ridl_policy_ppo.pth")
print("✅ Saved policy to ridl_policy_ppo.pth")

In [ ]:
# %%
# Cell 9 – Plot Metrics
epochs = list(range(1, len(imitation_loss_history)+1))
ppo_iters = list(range(1, len(ppo_reward_history)+1))

fig, axs = plt.subplots(2, 2, figsize=(12,10))

# Imitation loss
axs[0,0].plot(epochs, imitation_loss_history, marker='o')
axs[0,0].set_title('Imitation Pre-training Loss (MSE)')
axs[0,0].set_xlabel('Epoch')
axs[0,0].set_ylabel('MSE Loss')

# PPO Reward
axs[0,1].plot(ppo_iters, ppo_reward_history, marker='o')
axs[0,1].set_title('PPO Average Episode Reward')
axs[0,1].set_xlabel('PPO Iteration')
axs[0,1].set_ylabel('Average Reward')

# PPO Policy & Value Losses
axs[1,0].plot(ppo_iters, ppo_policy_loss_history, label='Policy Loss')
axs[1,0].plot(ppo_iters, ppo_value_loss_history, label='Value Loss')
axs[1,0].set_title('PPO Losses')
axs[1,0].set_xlabel('PPO Iteration')
axs[1,0].legend()

# PPO Entropy
axs[1,1].plot(ppo_iters, ppo_entropy_history, label='Entropy Loss')
axs[1,1].set_title('PPO Entropy')
axs[1,1].set_xlabel('PPO Iteration')
axs[1,1].set_ylabel('Entropy')

plt.tight_layout()
plt.show()


In [ ]:
!pip install ace_tools

In [ ]:
import pandas as pd

# 1) Load your CSV
df = pd.read_csv('calc.csv')

# 2) Drop the timestamp column
df_num = df.drop(columns=['timestamp','fz','fy','fx'])

# 3) Compute mean and standard deviation

stds  = df_num.std()
mean_std = stds.mean()
# 4) Display results

print("Column-wise standard deviations:\n", stds)
print("Column-wise means:\n", mean_std, "\n")


In [ ]:
#!/usr/bin/env python3
import os
import glob
import re
import pandas as pd
import numpy as np

# ─── USER PATHS ───────────────────────────────────────────────────────────────
PRED_DIR   = 'data/'
NIC_DIR    = 'data/VR-NVR/NIC'
ROS_DIR    = 'data/VR-NVR/ROS'
OUTPUT_DIR = 'data/VR-NVR/LGAF'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── FIXED PARAMS ─────────────────────────────────────────────────────────────
SUBJECTS = range(10)
DATASETS = ['VR', 'NVR']
TASKS    = ['Reference', 'Task1', 'Task2']
DT       = 1/50  # seconds between samples

for subj in SUBJECTS:
    for ds in DATASETS:
        for task in TASKS:
            # 1) find the prediction CSV, matching subject{subj}-RH|LH-{ds}_{task}.csv
            pred_glob = os.path.join(
                PRED_DIR,
                f'predictions_Subject-{subj}_*_subject{subj}-*-{ds}_{task}.csv'
            )
            matches = glob.glob(pred_glob)
            if not matches:
                print(f"[skip] no preds for S{subj}-{ds}-{task}")
                continue

            pred_csv = matches[0]
            fn       = os.path.basename(pred_csv)

            # extract the hand code ("RH" or "LH")
            m = re.search(
                rf'subject{subj}-(?P<hand>RH|LH)-{ds}_{task}\.csv$',
                fn
            )
            if not m:
                print(f"[warn] can't parse hand from {fn}; skipping")
                continue
            hand = m.group('hand')

            # 2) find the reference .easy file (note the literal "subject" in the name)
            ref_glob = os.path.join(
                NIC_DIR,
                f'Subject-{subj}',
                f'*_subject{subj}-{hand}-{ds}_{task}.easy'
            )
            refs = glob.glob(ref_glob)
            if not refs:
                print(f"[skip] no .easy for S{subj}-{hand}-{ds}-{task}")
                continue
            ref_file = refs[0]

            # 3) locate the ROS CSV
            ros_folder = os.path.join(ROS_DIR, f'subject{subj}-{hand}')
            ros_csv    = os.path.join(ros_folder, f'subject{subj}-{hand}-{ds}.csv')
            if not os.path.exists(ros_csv):
                print(f"[skip] no ROS data for S{subj}-{hand}-{ds}")
                continue

            print(f"\n→ Processing Subject {subj}, Hand={hand}, Dataset={ds}, Task={task}")

            # Step 1: load predictions & pick best logits
            df    = pd.read_csv(pred_csv)
            arsq  = [c for c in df if c.startswith('ARSQ_')  and '_Logit_C' in c]
            uma   = [c for c in df if c.startswith('UMACL_UWIST_') and '_Logit_C' in c]
            prefs = {c.rsplit('_Logit_C',1)[0] for c in arsq + uma}

            sel = pd.DataFrame(index=df.index)
            for pfx in prefs:
                cols = [f"{pfx}_Logit_C{i}" for i in range(5) if f"{pfx}_Logit_C{i}" in df]
                best = df[cols].idxmax(axis=1)
                sel[pfx] = best.str.extract(r'Logit_C(\d+)').astype(int)

            # zero-pad & sort ARSQ columns
            rm = {}
            for c in sel.columns:
                m2 = re.match(r'ARSQ_(\d+)$', c)
                rm[c] = f"ARSQ_{m2.group(1).zfill(2)}" if m2 else c
            sel.rename(columns=rm, inplace=True)
            sel = sel.reindex(sorted(sel.columns), axis=1)

            # Step 2: compute Va, Vb, Task_Ratio, F
            ars_cols = [c for c in sel if c.startswith('ARSQ_')]
            uma_cols = [c for c in sel if c.startswith('UMACL_UWIST_')]

            Va = sel[ars_cols].sum(axis=1)  / (len(ars_cols)  * 4)
            Vb = sel[uma_cols].sum(axis=1)  / (len(uma_cols) * 4)
            if 'Task_Logit_C0' not in df or 'Task_Logit_C2' not in df:
                raise KeyError(f"Missing Task_Logit_C0/2 in {pred_csv}")
            TR = df['Task_Logit_C0'] / df['Task_Logit_C2']
            Fm = (Vb / Va) * TR

            metrics = pd.DataFrame({
                'Va':         Va,
                'Vb':         Vb,
                'Task_Ratio': TR,
                'F':          Fm
            })

            # Step 3: build timestamps from .easy
            rdf       = pd.read_csv(ref_file, sep='\t', header=None)
            t0        = rdf.iloc[0, -1] / 1000.0
            n_samples = len(metrics)
            metrics['timestamp'] = (t0 + np.arange(n_samples) * DT).round(1)

            # Step 4: load ROS & round
            ros = pd.read_csv(ros_csv)
            ros['timestamp'] = ros['timestamp'].astype(float).round(1)

            # Step 5: merge & average
            merged = (
                pd.merge(ros, metrics, on='timestamp', how='inner')
                  .dropna(subset=['F'])
                  .groupby('timestamp', as_index=False)
                  .mean()
            )

            # Step 6: save
            out_path = os.path.join(
                OUTPUT_DIR,
                f'subject{subj}-{hand}-{ds}_{task}.csv'
            )
            merged.to_csv(out_path, index=False)
            print(f"  saved → {out_path}")

print("\n✅ All done!")
